In [57]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error as RMSE, r2_score as r2
import optuna
import warnings

warnings.filterwarnings('ignore')


In [19]:
# Load data
df = pd.read_csv('../data/processed/car_features.csv')

print('Shape', df.shape)
print('Columns', df.columns.tolist())


Shape (298, 10)
Columns ['Selling_Price', 'Present_Price', 'Driven_kms', 'Selling_type', 'Transmission', 'Owner', 'Car_Age', 'Fuel_Type_CNG', 'Fuel_Type_Diesel', 'Fuel_Type_Petrol']


In [ ]:
# Define target and features
TARGET = 'Selling_Price'
FEATURES = [ c for c in df.columns if c != TARGET]

X = df[FEATURES]
y = df[TARGET]

print(f"Target: {TARGET}")
print(f'Features: {FEATURES}')
print(f"Features Length: {len(FEATURES)}")

Target: Selling_Price
Features: ['Present_Price', 'Driven_kms', 'Selling_type', 'Transmission', 'Owner', 'Car_Age', 'Fuel_Type_CNG', 'Fuel_Type_Diesel', 'Fuel_Type_Petrol']
Features Length: 9


In [42]:
# Split, model, predict, 
def train_model (X, y, model=None, log_target=False, baseline=False):

    rmse_scores = []
    r2_scores = []

    # Initilize KFold
    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    for train_idx, test_idx in kf.split(X):

        # Split X 
        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]

        # Split y
        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        # Baseline model
        if baseline:
            # NB: baseline prediction = mean of training set
            y_pred = np.full_like(y_test, y_train.mean(), dtype=float)

        # ML model
        else:
            # Train model
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

        # Inverse transform if log target
        if log_target:
            y_test = np.expm1(y_test)
            y_pred = np.expm1(y_pred)

        # Metrics
        rmse_scores.append(RMSE(y_test, y_pred))
        r2_scores.append(r2(y_test, y_pred))

    return {
        "rmse_mean": float(np.round(np.mean(rmse_scores), 4)),
        "r2_mean": float(np.round(np.mean(r2_scores), 4))
    }


In [43]:
baseline_results = train_model(X, y, baseline=True)

baseline_results_log = train_model(
    X, 
    np.log1p(y), 
    baseline=True,
    log_target=True
)
print('----------- Baseline ---------------')
print(f'Baseline result without log: \n{baseline_results}')
print(f'Baseline result with log: \n{baseline_results_log}')

----------- Baseline ---------------
Baseline result without log: 
{'rmse_mean': 4.9402, 'r2_mean': -0.0132}
Baseline result with log: 
{'rmse_mean': 5.154, 'r2_mean': -0.1002}


In [44]:
# Linear Regression

# Without log
lr_results = train_model(
    X,
    y, 
    model=LinearRegression()
)

# With log
lr_results_log = train_model(
    X,
    np.log1p(y), 
    model=LinearRegression(),
    log_target=True
)

print('----------------- Linear Regression ---------------------')
print(f'Lr result without log: \n{lr_results}')
print(f'Lr result with log: \n{lr_results_log}')

----------------- Linear Regression ---------------------
Lr result without log: 
{'rmse_mean': 1.8399, 'r2_mean': 0.8579}
Lr result with log: 
{'rmse_mean': 20.3334, 'r2_mean': -53.4327}


In [53]:

# Without log
cr_results = train_model(
    X,
    y, 
    CatBoostRegressor(depth=6, iterations=400, learning_rate=0.05, verbose=False)
)

# With log
cr_results_log = train_model(
    X,
    np.log1p(y),
    CatBoostRegressor(depth=6, iterations=400, learning_rate=0.05, verbose=False),
    log_target=True
)

print('----------------- CAT BOOST ---------------------')
print(f'Cr result without log: \n{cr_results}')
print(f'Cr result with log: \n{cr_results_log}')

----------------- CAT BOOST ---------------------
Cr result without log: 
{'rmse_mean': 1.4259, 'r2_mean': 0.9039}
Cr result with log: 
{'rmse_mean': 1.551, 'r2_mean': 0.8816}


### Log Target: Linear Regression vs CatBoost

Linear Regression's performance collapsed when trained on log1p(Selling_Price) and
inverse-transformed (RMSE 20.33, R² -53.43), while raw-target Linear Regression
performed well (RMSE 1.84, R² 0.86). This isn't a transform failure in general —
CatBoost handled the same log target with only a small RMSE increase (1.43 → 1.55).

The difference: Linear Regression has no bound on its output, so a small error in
log-space (likely driven by high-leverage points from unscaled features like
Driven_kms) becomes an exponentially large error once expm1() is applied. CatBoost's
tree-based leaf outputs are bounded by values seen during training, so it can't
extrapolate to an extreme prediction the same way.

Since CatBoost on the raw target outperforms every other combination (RMSE 1.43,
R² 0.90), and the log transform adds no benefit here, raw Selling_Price will be
carried forward as the final model target — the right-skew flagged in EDA turned
out not to be problematic for a tree-based model.

In [ ]:
# CatBoost Hyperparameter tuning

def objective(trial):
    params = {
        'depth': trial.suggest_int('depth', 3, 8),
        'iterations': trial.suggest_int('iterations', 100, 600),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 20),
        'verbose': False,
        'random_state': 42
    }

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    rmse_scores = []

    for train_idx, test_idx in kf.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        model = CatBoostRegressor(**params)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        rmse_scores.append(RMSE(y_test, y_pred))

    return np.mean(rmse_scores)

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print('Best CV RMSE:', round(study.best_value, 4))
print('Best params:', study.best_params)

[I 2026-07-03 15:00:50,518] A new study created in memory with name: no-name-bcff5748-70eb-40d8-a5ac-9a5b8b794f56


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-07-03 15:00:58,404] Trial 0 finished with value: 1.503242998607504 and parameters: {'depth': 7, 'iterations': 372, 'learning_rate': 0.0696412277024887, 'l2_leaf_reg': 2.927905313841219, 'min_data_in_leaf': 16}. Best is trial 0 with value: 1.503242998607504.
[I 2026-07-03 15:01:04,798] Trial 1 finished with value: 1.575035047986031 and parameters: {'depth': 6, 'iterations': 571, 'learning_rate': 0.031380897583373535, 'l2_leaf_reg': 5.341026676291161, 'min_data_in_leaf': 13}. Best is trial 0 with value: 1.503242998607504.
[I 2026-07-03 15:01:08,207] Trial 2 finished with value: 1.7561964275230626 and parameters: {'depth': 3, 'iterations': 446, 'learning_rate': 0.011758011868945208, 'l2_leaf_reg': 2.2942144557962574, 'min_data_in_leaf': 15}. Best is trial 0 with value: 1.503242998607504.
[I 2026-07-03 15:01:12,968] Trial 3 finished with value: 1.6078649444617046 and parameters: {'depth': 7, 'iterations': 224, 'learning_rate': 0.14133476569180695, 'l2_leaf_reg': 7.91611493725647, '

In [60]:
tuned_params = {**study.best_params, 'verbose': False, 'random_state': 42}

tuned_results = train_model(X, y, CatBoostRegressor(**tuned_params))

print('----------------- Tuned CatBoost ---------------------')
print(f'Default CatBoost: {cr_results}')
print(f'Tuned CatBoost:   {tuned_results}')

----------------- Tuned CatBoost ---------------------
Default CatBoost: {'rmse_mean': 1.4259, 'r2_mean': 0.9039}
Tuned CatBoost:   {'rmse_mean': 1.1961, 'r2_mean': 0.9346}


In [62]:
results = pd.DataFrame({
    'Model': ['Mean Baseline', 'Mean Baseline (log)', 'Linear Regression',
              'Linear Regression (log)', 'CatBoost', 'CatBoost (log)',  "CatBoost (tuned)"],
    'RMSE': [baseline_results['rmse_mean'], baseline_results_log['rmse_mean'],
             lr_results['rmse_mean'], lr_results_log['rmse_mean'],
             cr_results['rmse_mean'], cr_results_log['rmse_mean'], tuned_results['rmse_mean']],
    'R2_SCORE': [baseline_results['r2_mean'], baseline_results_log['r2_mean'],
                 lr_results['r2_mean'], lr_results_log['r2_mean'],
                 cr_results['r2_mean'], cr_results_log['r2_mean'], tuned_results['r2_mean']]
}).sort_values('R2_SCORE', ascending=False)

print(results.to_string(index=False))

                  Model    RMSE  R2_SCORE
       CatBoost (tuned)  1.1961    0.9346
               CatBoost  1.4259    0.9039
         CatBoost (log)  1.5510    0.8816
      Linear Regression  1.8399    0.8579
          Mean Baseline  4.9402   -0.0132
    Mean Baseline (log)  5.1540   -0.1002
Linear Regression (log) 20.3334  -53.4327


In [63]:
final_model = CatBoostRegressor(**tuned_params)
final_model.fit(X, y)

importance = pd.DataFrame({
    'Feature': FEATURES,
    'Importance': final_model.get_feature_importance()
}).sort_values('Importance', ascending=False)

print(importance.to_string(index=False))

         Feature  Importance
   Present_Price   78.176566
         Car_Age   11.672286
      Driven_kms    3.712878
    Selling_type    2.609963
Fuel_Type_Diesel    1.773166
           Owner    1.053290
Fuel_Type_Petrol    0.862623
    Transmission    0.088531
   Fuel_Type_CNG    0.050699


In [64]:
import os
os.makedirs('../models', exist_ok=True)
final_model.save_model('../models/catboost_model.cbm')
print('Saved to ../models/catboost_model.cbm')

Saved to ../models/catboost_model.cbm


## Modelling Summary

### Evaluation strategy
5-fold cross-validation used as the primary evaluation method rather than a single
train/val split — with only 298 rows, a single 20% holdout (~60 rows) risked
unstable metrics depending on which rows landed in validation.

### Target transform decision
Raw Selling_Price outperformed log1p(Selling_Price) for CatBoost (RMSE 1.43 vs 1.55).
Linear Regression on the log target collapsed entirely (R² -53.43) due to unbounded
predictions being amplified by expm1() on high-leverage points — CatBoost's bounded
tree leaf outputs avoided this failure mode. Raw target carried forward.

### Hyperparameter tuning
Optuna, 50 trials, 5-fold CV RMSE as the objective. Tuning gave a meaningful
improvement over default parameters — RMSE 1.4259 → 1.1961 (-16.1%), R² 0.9039 →
0.9346.

Best parameters: depth=3, iterations=527, learning_rate=0.1343, l2_leaf_reg=6.74,
min_data_in_leaf=11.

### Results

| Model | RMSE | R² |
|-------|------|-----|
| CatBoost (tuned) | 1.1961 | 0.9346 |
| CatBoost | 1.4259 | 0.9039 |
| CatBoost (log) | 1.5510 | 0.8816 |
| Linear Regression | 1.8399 | 0.8579 |
| Mean Baseline | 4.9402 | -0.0132 |
| Mean Baseline (log) | 5.1540 | -0.1002 |
| Linear Regression (log) | 20.3334 | -53.4327 |

### Feature importance
Present_Price dominates (78.2%), followed by Car_Age (11.7%) — together ~90% of
the model's decisions. Consistent with the 0.879 Present_Price correlation and
clear age-price relationship found in EDA. Remaining features (Driven_kms,
Selling_type, Fuel_Type, Owner, Transmission) contribute marginally.

### Final model
CatBoost, tuned parameters, fit on the full 298-row dataset. Saved to
../models/catboost_model.cbm.

### Next step
Evaluation — residual analysis, error distribution by price range, and business
interpretation of where the model is least reliable.